# Start here: a small native AUTOENCODIX experiment

Run the cells in order with the **VAE beta control (Python 3.11)** kernel.
This CPU example generates a circle dataset, trains one VAE for three epochs,
and exports posteriors, metrics and plots. No TCGA files, external drive,
API keys or pretrained checkpoints are needed. It is a technical smoke test,
not evidence of convergence or biological performance.

Setup from a terminal is documented in [README.md](README.md).


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "beta_vae_experiments").is_dir():
    raise RuntimeError("Open this notebook from the cloned repository root.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Select the Python 3.11 project kernel.")
print("Repository:", ROOT)
print("Python:", sys.executable)


## 1. Inspect the experiment

All settings live in JSON. The example uses AUTOENCODIX `logistic-mid`
annealing. For fixed beta use `no-annealing`; `3phase-linear` is another
native option. Existing completed runs are reused; change the experiment name
and output directory to run a separate experiment.


In [ ]:
import json
from beta_vae_experiments.new_beta_varix_experiments import load_experiment_config, preflight_experiment

CONFIG = ROOT / "beta_vae_experiments/user_interface/quickstart.json"
print(CONFIG.read_text())
preflight_experiment(CONFIG)


## 2. Train and evaluate

This cell starts the small CPU run. Larger experiments should use their own JSON configuration.


In [ ]:
from beta_vae_experiments.new_beta_varix_experiments import run_experiment

results = run_experiment(CONFIG)
if results.empty or not results["status"].eq("ok").all():
    raise RuntimeError("Training failed: inspect the error columns below and terminal output.")
display(results)


## 3. Inspect results

The output directory contains model checkpoints, aligned posterior means and log-variances, metric tables and diagnostic plots. Undefined metrics can occur for degenerate or very small representations; inspect their status fields.


In [ ]:
import pandas as pd
from IPython.display import Image, display

output = Path(load_experiment_config(CONFIG)["output_dir"])
print("Results:", output)
metric_files = sorted(output.rglob("scalar_metrics.csv"))
if metric_files:
    metrics = pd.read_csv(metric_files[0])
    display(metrics.head(20))
for figure in sorted((output / "metric_analysis").rglob("*.png"))[:4]:
    print(figure.name)
    display(Image(filename=str(figure)))


## Next steps

- Edit a copy of `quickstart.json` for more epochs, seeds, beta values or synthetic datasets.
- TCGA needs separately downloaded and prepared data; see the repository README.
- `analysis.ipynb` and `additional_downstream_tasks.ipynb` reproduce research analyses and require their saved data artifacts. They are not fresh-clone demos.
- Curvature and spectral studies are optional analyses, not required to train this example.
